# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools

## ✅ Solution — what this notebook implements

This notebook is a **self-contained** single-agent smart assistant (pure
standard library, no installs — runs top-to-bottom in Colab/Jupyter).

**Required**
- **Agent logic** + **conditional routing** by intent — `agent()` below.
- **Tool integration** — Calculator and Keyword Extractor.
- **Basic error handling** — every tool call is guarded; failures return
  `{"type": "error", "result": ...}` instead of crashing.
- **Structured JSON output** — every response is `{"type": ..., "result": ...}`.

**Bonus (all three)**
- **Improved routing** — a safe calculator (no raw `eval`), digit-gating so
  *"what is your name?"* is **not** sent to the calculator, and
  keyword/stats checks ordered ahead of arithmetic.
- **Logging** — Python's `logging` records every routing decision and result.
- **More tools** — a **Text-Stats** tool (word/char/sentence counts) and a
  **Reverse-Text** tool, both fully routed.

Run the cells top to bottom (or *Runtime → Run all*); the final cells exercise
every requirement with live output.

In [1]:
# 📦 Imports & logging setup (bonus: logging)
import ast
import logging
import operator
import re
import sys

# Log to stdout (not stderr) so routing decisions show as normal cell output
# rather than red error text; force=True re-applies config even if the host
# (e.g. Colab) already configured the root logger.
logging.basicConfig(stream=sys.stdout, level=logging.INFO,
                    format="%(levelname)s | %(message)s", force=True)
log = logging.getLogger("agent")

In [2]:
# 🛠️ TOOL 1: Calculator (safe — no raw eval)

# Only these operators are allowed; arbitrary names / function calls are
# rejected, so this is safe to run on user input (unlike bare eval()).
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod, ast.Pow: operator.pow,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval(node):
    if isinstance(node, ast.Expression):
        return _eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) \
            and not isinstance(node.value, bool):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("unsupported expression")


def calculator(expression: str):
    """Evaluate a mathematical expression safely. Raises on bad input."""
    expr = expression.replace("^", "**").strip()
    value = _eval(ast.parse(expr, mode="eval"))
    if isinstance(value, float) and value.is_integer():
        value = int(value)            # 4.0 -> 4 for tidy output
    return value


# quick check
print(calculator("2 + 3 * 4"), calculator("(10 - 4) / 2"), calculator("2 ^ 5"))

14 3 32


In [3]:
# 🛠️ TOOL 2: Keyword Extractor (frequency-ranked, stop-words removed)

_STOPWORDS = set("""
a an and are as at be by for from has have how i in is it its of on or that the
to was were what when where which who why will with you your we they this these
those there here do does did can could should would may might must not into over
""".split())
_WORD_RE = re.compile(r"[A-Za-z][A-Za-z0-9'-]*")


def extract_keywords(text: str, top_k: int = 5) -> list:
    """Return up to ``top_k`` salient keywords, most frequent first."""
    counts = {}
    for m in _WORD_RE.finditer(text.lower()):
        w = m.group(0)
        if w in _STOPWORDS or len(w) < 2:
            continue
        counts[w] = counts.get(w, 0) + 1
    # rank by frequency, ties broken alphabetically for determinism
    ranked = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    return [w for w, _ in ranked[:top_k]]


print(extract_keywords("Artificial Intelligence is transforming industries"))

['artificial', 'industries', 'intelligence', 'transforming']


In [4]:
# 🛠️ TOOL 3 (bonus): Text statistics

def text_stats(text: str) -> dict:
    """Count words, characters, and sentences."""
    words = len(text.split())
    characters = len(text)
    sentences = len([s for s in re.split(r"[.!?]+", text) if s.strip()])
    if sentences == 0 and text.strip():
        sentences = 1
    return {"words": words, "characters": characters, "sentences": sentences}


print(text_stats("Hello there. How are you?"))

{'words': 5, 'characters': 25, 'sentences': 2}


In [5]:
# 🛠️ TOOL 4 (bonus): Reverse text

def reverse_text(text: str) -> str:
    """Return the text reversed."""
    return text[::-1]


print(reverse_text("agentic pipeline"))

enilepip citnega


## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

*(Routing below also handles the two bonus tools and gates the calculator on
the presence of a number, so plain questions like "what is your name?" fall
through to the general responder instead of failing in the calculator.)*

In [6]:
# 🤖 AGENT FUNCTION  ->  returns {"type": ..., "result": ...}

_CALC_WORDS = re.compile(r"\b(calculate|compute|evaluate|what\s+is|sum|product)\b", re.I)
_ARITHMETIC = re.compile(r"\d\s*[-+*/%^]\s*\d")
_KEYWORD_WORDS = re.compile(r"\b(keywords?|key\s*terms?|extract|tags?|topics?)\b", re.I)
_STATS_WORDS = re.compile(r"\b(word\s*count|count\s+(the\s+)?(words?|characters?|chars?)|"
                          r"how\s+many\s+(words?|characters?)|number\s+of\s+(words?|characters?)|"
                          r"statistics|text\s+stats)\b", re.I)
_REVERSE_WORDS = re.compile(r"\breverse\b", re.I)
_MARKERS = (" from:", " in:", " of:", " from ", " in ", " of ", ":")


def _content(query: str, triggers: str) -> str:
    """Pull the text a request is *about* (after 'in:'/'from:' etc.)."""
    low = query.lower()
    idxs = [(low.find(m), len(m)) for m in _MARKERS if low.find(m) != -1]
    if idxs:
        i, ln = min(idxs)
        return query[i + ln:].strip()
    return re.sub(triggers, " ", query, flags=re.I).strip() or query.strip()


def _expression(query: str) -> str:
    """Pull the arithmetic expression out of a natural-language query."""
    cands = [m.group(0).strip() for m in re.finditer(r"[-+*/%^().\d\s]+", query)]
    best = max((c for c in cands if any(ch.isdigit() for ch in c)), key=len, default="")
    return re.sub(r"[\s+\-*/%(.]+$", "", best).strip()


def agent(query: str) -> dict:
    q = query.strip()
    low = q.lower()
    has_digit = any(ch.isdigit() for ch in low)
    try:
        # --- conditional routing (keywords / stats / reverse before math) ---
        if _KEYWORD_WORDS.search(low):
            route = "keywords"
            result = {"type": "keywords",
                      "result": extract_keywords(_content(q, r"\b(extract|keywords?|tags?|topics?)\b"))}
        elif _STATS_WORDS.search(low):
            route = "stats"
            result = {"type": "stats",
                      "result": text_stats(_content(q, r"\b(word|count|words|characters|chars|how|many|number|of|statistics|stats|text|the)\b"))}
        elif _REVERSE_WORDS.search(low):
            route = "reverse"
            result = {"type": "reversed",
                      "result": reverse_text(_content(q, r"\b(reverse|the|text|string|this)\b"))}
        elif _ARITHMETIC.search(low) or (_CALC_WORDS.search(low) and has_digit):
            route = "calculator"
            result = {"type": "calculation", "result": calculator(_expression(q))}
        else:
            route = "general"
            result = {"type": "general",
                      "result": (f"This looks like a general question. I don't have a language "
                                 f"model for open-ended answers, but I can calculate math, extract "
                                 f"keywords, count words, or reverse text. (You asked: {q!r})")}
        log.info("query=%r -> route=%s", q, route)
        return result
    except Exception as exc:                      # basic error handling
        log.error("query=%r failed: %s", q, exc)
        return {"type": "error", "result": f"Could not process query: {exc}"}

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```
*(bonus tools extend the set with `"stats"` and `"reversed"`.)*

In [7]:
# 🧪 Test Cases

queries = [
    # the three required cases
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    # bonus tools
    "Count the words in: the quick brown fox jumps",
    "Reverse hello world",
    # error handling
    "Calculate 8 / 0",
]

for q in queries:
    response = agent(q)          # logs the routing decision (see INFO lines)
    print("Query   :", q)
    print("Response:", response)
    print("-" * 50)

INFO | query='Calculate 20 + 5' -> route=calculator
Query   : Calculate 20 + 5
Response: {'type': 'calculation', 'result': 25}
--------------------------------------------------
INFO | query='Extract keywords from Artificial Intelligence is transforming industries' -> route=keywords
Query   : Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['artificial', 'industries', 'intelligence', 'transforming']}
--------------------------------------------------
INFO | query='What is machine learning?' -> route=general
Query   : What is machine learning?
Response: {'type': 'general', 'result': "This looks like a general question. I don't have a language model for open-ended answers, but I can calculate math, extract keywords, count words, or reverse text. (You asked: 'What is machine learning?')"}
--------------------------------------------------
INFO | query='Count the words in: the quick brown fox jumps' -> route=stats
Query   : 

In [8]:
# 🎯 Interactive Mode
# Set RUN_INTERACTIVE = True and run this cell to chat with the agent.
# (Left False so "Run all" doesn't block waiting for input.)

RUN_INTERACTIVE = False

if RUN_INTERACTIVE:
    while True:
        try:
            user_input = input("Enter query (type 'exit' to stop): ")
        except (EOFError, KeyboardInterrupt):
            break
        if user_input.lower() == "exit":
            break
        print("Response:", agent(user_input))

## 📋 Requirements checklist

| Requirement | Status | Where |
|---|---|---|
| Understands queries & routes by intent | ✅ | `agent()` conditional routing |
| Math → Calculator tool | ✅ | `calculator()` (safe, no raw `eval`) |
| Keyword extraction → Keyword tool | ✅ | `extract_keywords()` |
| General queries → direct response | ✅ | `agent()` else-branch |
| Tool integration | ✅ | tools called from `agent()` |
| Structured JSON output | ✅ | every return is `{type, result}` |
| Basic error handling | ✅ | `try/except` → `{"type": "error", ...}` |
| **Bonus:** improved routing | ✅ | safe eval, digit-gating, ordered checks |
| **Bonus:** logging | ✅ | `logging` on every decision/result |
| **Bonus:** more tools | ✅ | `text_stats()` + `reverse_text()` |
| Interactive mode | ✅ | guarded REPL cell above |